In [ ]:
import pandas as pd 
import numpy as np
df = pd.read_csv('../Dataset/processed/final_dataset_preprocessed.csv')

In [2]:
from sklearn.preprocessing import StandardScaler

feature_cols = df.columns.drop('target')

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [3]:
import numpy as np

def create_sequences(df, window=7):
    X, y = [], []
    
    data = df.values
    
    for i in range(len(data) - window):
        X.append(data[i:i+window, :-1])  # all features except target
        y.append(data[i+window, -1])     # target
    
    return np.array(X), np.array(y)

In [4]:
X, y = create_sequences(df, window=7)

print(X.shape)  # (samples, 7, features)
print(y.shape)

(9125, 7, 29)
(9125,)


In [5]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [6]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        
        out = self.dropout(out)
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        
        return self.sigmoid(out)

d:\IoT_Precision_Agriculture\IOT_VENV\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = LSTMModel(input_size=X.shape[2]).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Using device: cuda


In [10]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

epochs = 30

for epoch in range(epochs):
    model.train()
    
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.5802
Epoch 2, Loss: 0.5686
Epoch 3, Loss: 0.5550
Epoch 4, Loss: 0.5420
Epoch 5, Loss: 0.5264
Epoch 6, Loss: 0.5112
Epoch 7, Loss: 0.4937
Epoch 8, Loss: 0.4758
Epoch 9, Loss: 0.4567
Epoch 10, Loss: 0.4374
Epoch 11, Loss: 0.4171
Epoch 12, Loss: 0.3974
Epoch 13, Loss: 0.3762
Epoch 14, Loss: 0.3553
Epoch 15, Loss: 0.3341
Epoch 16, Loss: 0.3149
Epoch 17, Loss: 0.2955
Epoch 18, Loss: 0.2778
Epoch 19, Loss: 0.2608
Epoch 20, Loss: 0.2451
Epoch 21, Loss: 0.2326
Epoch 22, Loss: 0.2209
Epoch 23, Loss: 0.2095
Epoch 24, Loss: 0.2006
Epoch 25, Loss: 0.1932
Epoch 26, Loss: 0.1866
Epoch 27, Loss: 0.1802
Epoch 28, Loss: 0.1750
Epoch 29, Loss: 0.1724
Epoch 30, Loss: 0.1677


In [11]:
model.eval()

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
    preds = model(X_test_t).cpu().numpy()

preds = (preds > 0.5).astype(int)

In [12]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.806027397260274
              precision    recall  f1-score   support

         0.0       0.55      0.87      0.67       418
         1.0       0.95      0.79      0.86      1407

    accuracy                           0.81      1825
   macro avg       0.75      0.83      0.77      1825
weighted avg       0.86      0.81      0.82      1825

